# L2 · Step 1 again: teacher answers from the new 7B (GPU T4 ×2)

Task **L2**, step 1, from `Essential_task_delegation/Next_Steps_Roadmap.md`. Goal **O4**: a distilled 3B model. A named copy of `l2-step1-teacher-rationales.ipynb`, which stays as it ran on 19 Oct.

**Why again:** L0 step 2's verdict (28 Oct) kept the new 7B, `AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP`, so it replaces the old 7B as the 3B's teacher. The first run's answers came from the old 7B.

**What changed from the first run, and why:**
- **The teacher:** the new 7B.
- **Full size** (1036 px for a 1024 px image) instead of the 512-token cap: the new 7B was trained at full size, and L0 measured it there.
- **Greedy decoding,** with every sampling setting given explicitly, instead of sampling at 0.2. The teacher answers in one sentence, so sampling adds noise, not useful variety; L0 also scored it greedy.
- **The plain training prompt,** without the "think step by step" instruction. The 7B ignored it in the first run, and without it the teacher answers the same prompt the 3B will see.
- **The first run's entries minus road counting:** its 4,000 (seed 42) without the 735 road-counting entries. Their labels are exact percentages such as 5.22%, and none of the first run's 735 answers matched one. 3,265 entries, about an hour less of GPU.
- **No push to a Kaggle dataset:** the first run's push printed "Pushed" after it had failed. Step 2 attaches this notebook's output instead.
- **Resuming reuses only this teacher's answers:** each progress row carries the teacher's ID.
- The answer check (Cell 4) is unchanged.

**Expectation, fixed before the run:**
- The same 3,265 entries as the first run's non-road part (Cell 4 asserts it).
- More kept answers for disaster type (first run 89.4%) and relational reasoning (13.3%). Part of that is memory: L0's top-up trained on these training entries again.
- Building counting, bearing bodies and land use within about 3 points of the first run (15.6%, 42.7% and 50.9%).
- At least the first run's 1,085 kept answers in all.
- About 5.5 h on T4 ×2: Cell 4 estimates 5.3 h for the answers (3,085 entries with two images, 180 with one), plus the download and loading. The first run took 2.0 h at the 512-token cap.

The first run's numbers are a reference, not a controlled comparison: the teacher, image size, decoding and prompt all changed, each for the reason above.

**Dry run on your PC (28 Oct),** with the first run's output manifest standing in for the training data and a stand-in for the model: Cell 4 reproduced the first run's 4,000 entries exactly and left 3,265 after removing road counting; the unchanged answer check re-kept exactly the first run's 1,085 answers; resuming ignored another teacher's rows and reused this teacher's; Cell 6 wrote all five files. The first run's longest answer was 177 characters, well inside the 96-token limit.

### How to run
1. Sign in as account 1 (`abrarmohammedtanzim`, which owns the DisasterM3 mirror). **Create → New Notebook → File → Import Notebook → Browse**, pick this file, then **Import**.
2. **Session options:** GPU **T4 x2**, Internet **On**.
3. **Input → + Add Input:** `abrarmohammedtanzim/disasterm3-mirror`.
4. No secret is needed: the new 7B's repo is public. A `HF_TOKEN` secret only speeds up the 16 GB download.
5. **Smoke test:** in Cell 2 set `SMOKE_TEST = True`, then **Run → Run All**, about 15–20 min with the download. Check:
   - Cell 3: `full size checked (1036 px)`;
   - Cell 4: `Same entries as the first run: 4,000 … 735 road-counting entries left out` and `self-checks passed`;
   - Cell 5: three answers, each one sentence, some matching their label;
   - Cell 6: a table by task, then `✓ Wrote /kaggle/working/L2S1-new-teacher`.
6. Set `SMOKE_TEST = False`. **Save Version → Save & Run All (Commit) → Save.** It stops cleanly at 11 h; to continue, attach that version's output and run again.
7. When it's finished, download the `L2S1-new-teacher` folder to `Champion_Task/L2-step1-results/new-teacher/`, and save the run notebook over this file.

In [ ]:
# ── Cell 1: Environment (Kaggle's own torch; transformers only if too old for Qwen2.5-VL) ──
# The original's pins (torch 2.4.1, transformers 4.49, bitsandbytes 0.45) have no wheels for Kaggle's
# Python 3.13, so this cell pins nothing. Same approach as h1-v2-counts-as-text.ipynb.
import importlib.metadata as _md
import subprocess
import sys

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

def _older(have, want):
    nums = lambda v: tuple(int(x) for x in v.split(".")[:2] if x.isdigit())
    return nums(have) < nums(want)

if _older(_version("transformers"), "4.49") or _version("accelerate") == "0":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.49", "accelerate"], check=True)

import torch
import transformers
print(f"✓ torch {torch.__version__}, transformers {transformers.__version__}, python {sys.version.split()[0]}")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 2, "Pick GPU T4 x2: the 7B teacher in fp16 needs both T4s"

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────
import os, json, random, re, time, glob, shutil, difflib
from pathlib import Path
from datetime import datetime

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"   # same original manifest the 7B notebook trained on

TEACHER_MODEL_ID = "AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP"   # L0 step 2's new 7B, kept on 28 Oct (public)
# Full size, as L0 trained and scored the new 7B: its repo keeps the stock Qwen2.5-VL processor (Cell 3 checks it).

GEN_MAX_NEW_TOKENS = 96        # the teacher answers in one sentence; a longer answer is cut and then fails the check
BATCH = 8                      # examples per generate() call; halved automatically on out-of-memory

SMOKE_TEST = False             # True = 16 entries, to check the setup and read a few answers
# The entries: the first run's subset, drawn the same way (seed 42), minus road counting. Road-counting labels
# are exact percentages such as 5.22%, and none of the first run's 735 answers matched one.
FIRST_RUN_SIZE = 4000
RANDOM_SEED = 42
FIRST_RUN_ELIGIBLE, FIRST_RUN_CHECKABLE = 55764, 40233    # Cell 4 checks that the subset is drawn from the same pool
SKIPPED = {"Road Damage Counting"}
# Tasks whose answer can be checked against the label (Cell 4). Captions and restoration advice are
# long free text with no single answer, so they keep their plain labels.
CHECKABLE = {"Building Damage Counting", "Road Damage Counting", "Disaster Type Recognition",
             "Disaster Bearing Bodies Recognition", "Disaster Scene Recognition", "relational reasoning"}
# The first run (19 Oct; the old 7B, the 512-token cap, sampling at 0.2, "think step by step"): answers, kept
FIRST_RUN = {"Building Damage Counting": (1445, 225), "Disaster Bearing Bodies Recognition": (776, 331),
             "Disaster Scene Recognition": (694, 353), "Disaster Type Recognition": (170, 152),
             "Road Damage Counting": (735, 0), "relational reasoning": (180, 24)}

OUTPUT_DIR = Path("/kaggle/working/L2S1-new-teacher")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_PATH = OUTPUT_DIR / "teacher_cot_progress.jsonl"   # incremental, resumable
FINAL_MANIFEST_PATH = OUTPUT_DIR / "disasterm3_distillation_cot_train.json"   # step 2's DISTILL_MANIFEST_PATH
TIME_LIMIT_HOURS = 11.0  # stop cleanly before Kaggle's 12h wall (loading takes about 10 minutes of it)

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

print(f"Teacher: {TEACHER_MODEL_ID}")
print(f"Manifest: {MANIFEST_PATH}")
print("[SMOKE TEST] 16 entries" if SMOKE_TEST else "The first run's entries, minus road counting")
print(f"Devices: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")

In [ ]:
# ── Cell 3: Load the teacher in fp16, split across both T4s, and check the full size ───────────────
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from PIL import Image

print(f"Loading teacher {TEACHER_MODEL_ID} in fp16 across {torch.cuda.device_count()} GPUs...")
processor = AutoProcessor.from_pretrained(TEACHER_MODEL_ID, token=hf_token())   # the stock processor: full size
processor.tokenizer.padding_side = "left"              # batched generation needs left padding
# transformers 4.56 renamed torch_dtype to dtype; older versions may ignore dtype and load fp32 (30 GB)
dtype_arg = "torch_dtype" if _older(transformers.__version__, "4.56") else "dtype"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    TEACHER_MODEL_ID,
    **{dtype_arg: torch.float16},
    device_map="auto",
    attn_implementation="sdpa",
    token=hf_token(),
)
model.eval()

# Full size on a 1024 x 1024 test image, from the processor's grid of 14 px patches (as in L0's re-score notebook)
_, h, w = processor.image_processor(images=[Image.new("RGB", (1024, 1024), (90, 120, 80))],
                                    return_tensors="pt")["image_grid_thw"][0].tolist()
assert (h * 14, w * 14) == (1036, 1036), (f"The teacher would see {h * 14} x {w * 14} px, not 1036: "
                                          "check the processor's min_pixels and max_pixels.")
print(f"Teacher loaded ({dtype_arg}=float16), full size checked (1036 px); GPU memory "
      f"{[f'{torch.cuda.memory_allocated(i) / 1e9:.1f} GB' for i in range(torch.cuda.device_count())]}")

In [ ]:
# ── Cell 4: Load manifest, pick the subset, build the prompt, and the answer check ──
from collections import defaultdict
from PIL import Image

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from {MANIFEST_PATH}")

PATH_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff")

def resolve_image_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "box_train_images" / rel_path,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "box_train_images" / "box_train_images" / filename,
        DATA_ROOT / "masks" / "masks" / filename,
        Path("/tmp") / rel_path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return None

def answer_text(entry):
    ans = entry.get("training_answer", "")
    if isinstance(ans, list):
        ans = ans[0] if ans else ""
    return str(ans).strip()

def eligible(entry):
    # Same exclusions the 7B training notebook applied: drop segmentation /
    # path-like-answer entries and anything whose images don't resolve.
    if entry.get("task") == "Referring Expression Segmentation":
        return False
    ans = answer_text(entry)
    if not ans or ans.lower().endswith(PATH_SUFFIXES):
        return False
    if "image_path" in entry:
        return resolve_image_path(entry["image_path"]) is not None
    return (resolve_image_path(entry.get("pre_image_path", "")) is not None
            and resolve_image_path(entry.get("post_image_path", "")) is not None)

eligible_entries = [e for e in raw_data if eligible(e)]
print(f"Eligible for the distillation manifest: {len(eligible_entries):,} / {len(raw_data):,}")

# The first run's subset, drawn exactly as it was (seed 42, 4,000 of the checkable entries), minus road counting
checkable = [i for i, e in enumerate(eligible_entries) if e.get("task") in CHECKABLE]
random.seed(RANDOM_SEED)
first_run_indices = set(random.sample(checkable, k=min(FIRST_RUN_SIZE, len(checkable))))
subset_indices = {i for i in first_run_indices if eligible_entries[i]["task"] not in SKIPPED}
assert (len(eligible_entries), len(checkable)) == (FIRST_RUN_ELIGIBLE, FIRST_RUN_CHECKABLE), \
    "the manifest or the images differ from the first run's, so its subset can't be reproduced"
assert len(first_run_indices - subset_indices) == FIRST_RUN["Road Damage Counting"][0], "not the first run's subset"
print(f"Same entries as the first run: {len(first_run_indices):,} drawn with seed {RANDOM_SEED} from "
      f"{len(checkable):,} checkable entries; {len(first_run_indices - subset_indices):,} road-counting entries left out.")
if SMOKE_TEST:
    subset_indices = set(random.Random(RANDOM_SEED).sample(sorted(subset_indices), 16))
two_image = sum("image_path" not in eligible_entries[i] for i in subset_indices)
print(f"Answering {len(subset_indices):,} entries ({two_image:,} with two images): about "
      f"{(two_image * 6.0 + (len(subset_indices) - two_image) * 3.2) / 3600:.1f} h at full size. "
      f"Everything else keeps its plain label; step 2 falls back to it.")

def build_gen_messages(entry):
    """-> (chat messages, PIL images): the images first, then the training prompt as it is, with no added
    instruction. The stock processor keeps each image at full size (Cell 3 checks it)."""
    keys = ["image_path"] if "image_path" in entry else ["pre_image_path", "post_image_path"]
    images = [Image.open(resolve_image_path(entry[k])).convert("RGB") for k in keys]
    prompt = entry.get("prompts", "")
    if isinstance(prompt, list):
        prompt = prompt[0]
    content = [{"type": "image"} for _ in images] + [{"type": "text", "text": prompt}]
    return [{"role": "user", "content": content}], images

NUMBER = re.compile(r"\d+(?:\.\d+)?")
NUMBER_WORDS = {w: str(i) for i, w in enumerate(
    "zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen "
    "seventeen eighteen nineteen twenty".split())}       # small counts are often written as words
GENERIC = {"area", "areas"}                              # 'Residential area' is also said 'residential zones'
ALIASES = {"volcano": ["volcanic"]}
CATEGORY_TASKS = ("Disaster Type Recognition", "Disaster Bearing Bodies Recognition", "Disaster Scene Recognition")

def final_line(text):
    lines = [line.strip() for line in str(text).strip().splitlines() if line.strip()]
    return lines[-1] if lines else ""

def numbers(text):
    """The numbers in a text, with number words ('three') read as digits."""
    text = re.sub(r"\b(" + "|".join(NUMBER_WORDS) + r")\b", lambda m: NUMBER_WORDS[m.group(1)], str(text).lower())
    return {float(x) for x in NUMBER.findall(text)}

def canon(name):
    """One key per category: lower case, spaces collapsed, 'area(s)' dropped, a final 's' dropped,
    so 'Forest', 'Forests' and 'Forest  ' are the same category."""
    words = [w for w in str(name).lower().split() if w not in GENERIC]
    key = " ".join(words) or " ".join(str(name).lower().split())
    return key[:-1] if key.endswith("s") else key

def mentioned(option, text):
    """Is this option named in the text? Tolerates plurals and wordings like 'forested' or 'residential zones';
    whole words only, so 'port' is not found in 'airport', nor 'dams' in 'damaged'."""
    words = [w for w in str(option).lower().split() if w not in GENERIC] or str(option).lower().split()
    phrase = " ".join(words)
    phrase = phrase[:-1] if phrase.endswith("s") else phrase
    names = [phrase] + ALIASES.get(str(option).lower(), [])
    return any(re.search(rf"\b{re.escape(n)}(?:s|es|ed|ing)?\b", text) for n in names)

# Every category each task uses, across all its questions (a question shows only 8 of them). The dataset's
# option lists hold an empty name, 'None', stray notes in brackets and variants with spaces: those are dropped.
VOCAB = defaultdict(dict)                                # task -> {category key: one of its names}
for e in raw_data:
    if e.get("task") in CATEGORY_TASKS:
        for o in map(str, e.get("options_list") or []):
            key = canon(o)
            if key and key != "none" and "(" not in o:
                VOCAB[e["task"]].setdefault(key, o)
print(f"Categories per task: { {t: len(v) for t, v in VOCAB.items()} }")

def matches_label(entry, rationale):
    """True if the answer's last line gives the labelled answer: the same number for counting; for disaster type,
    bearing bodies and land use, exactly the labelled categories among every category the task uses (so an extra
    one counts as wrong); for relational reasoning, the labelled option as the closest one, at least 0.8 similar.
    Anything unclear counts as no match. On train_release.json, 99.9-100% of the labels themselves pass, per task
    (checked 19 Oct)."""
    answer, truth, task = final_line(rationale).lower(), entry.get("ground_truth"), entry.get("task")
    if task in ("Building Damage Counting", "Road Damage Counting"):
        return numbers(answer) == numbers(truth)
    if task == "relational reasoning":
        options = [str(o).lower().rstrip(".") for o in entry["options_list"]]
        score = {o: difflib.SequenceMatcher(None, answer.rstrip("."), o).ratio() for o in options}
        best = max(score, key=score.get)
        return best == str(truth).lower().rstrip(".") and score[best] >= 0.8
    if task in CATEGORY_TASKS:
        said = {key for key, name in VOCAB[task].items() if mentioned(name, answer)}
        return said == {canon(t) for t in str(truth).split(",") if canon(t)}
    return False

# Self-checks on label formats copied from train_release.json
_bdc = {"task": "Building Damage Counting", "ground_truth": 231, "options_list": [231, 323, 277, 139, 185]}
_bdc3 = {"task": "Building Damage Counting", "ground_truth": 3, "options_list": [3, 5, 7, 9, 11]}
_road = {"task": "Road Damage Counting", "ground_truth": "5.22%", "options_list": ["1.11%", "4.72%", "5.22%"]}
_dtr = {"task": "Disaster Type Recognition", "ground_truth": "explosion"}
_volcano = {"task": "Disaster Type Recognition", "ground_truth": "volcano"}
_bbr = {"task": "Disaster Bearing Bodies Recognition", "ground_truth": "Buildings, Roads, Forest, Open-space ground"}
_bbr2 = {"task": "Disaster Bearing Bodies Recognition", "ground_truth": "Farmland, Forest"}
_dsr = {"task": "Disaster Scene Recognition", "ground_truth": "Parking, Commercial area"}
_rel = {"task": "relational reasoning",
        "ground_truth": "The destroyed buildings in the red box are below the red intact building in the blue box.",
        "options_list": ["The destroyed buildings in the red box are below the red intact building in the blue box",
                         "The cluster of damaged buildings in the red box is situated directly below the intact "
                         "road in the blue box"]}
assert matches_label(_bdc, "I count the roofs.\nThe total number of undamaged buildings after the disaster is 231.")
assert not matches_label(_bdc, "The total number of undamaged buildings after the disaster is 230.")
assert matches_label(_bdc3, "A total of three buildings were completely destroyed in the disaster.")
assert matches_label(_road, "Roads are narrow.\nThe undamaged road area represents 5.22% of the total image area.")
assert matches_label(_dtr, "Collapsed roofs around a crater.\nThis area has been affected by an explosion.")
assert not matches_label(_dtr, "This area has been affected by a fire.")
assert matches_label(_volcano, "This region has been affected by a volcanic disaster.")
assert matches_label(_bbr, "The damaged objects include buildings, roads, forested areas, and open-space ground.")
assert not matches_label(_bbr, "The damaged objects include buildings and roads.")
assert not matches_label(_bbr2, "The critical objects that show disaster-related damage include farmland, forest, and roads.")
assert matches_label(_bbr2, "The critical objects that show disaster-related damage include farmland and forest.")
assert matches_label(_dsr, "The land-use zones present include parking areas and commercial zones.")
assert not matches_label(_dsr, "The land-use zones present include parking areas and residential zones.")
assert matches_label(_rel, "The destroyed buildings in the red box are below the red intact building in the blue box.")
assert not matches_label(_rel, "The cluster of damaged buildings in the red box is situated directly below the "
                               "intact road in the blue box.")
assert not matches_label({"task": "disaster caption", "ground_truth": "x"}, "x")
print("✓ Prompt and answer check ready (self-checks passed)")

In [ ]:
# ── Cell 5: Generation loop (greedy, batched, resumable, written as it goes) ───────
from tqdm.auto import tqdm

# Continue this run, and only this run: copy in an earlier version's progress from this notebook's own output folder.
# Rows from another teacher (the first run's file has none of this field) are never reused.
previous = sorted(glob.glob("/kaggle/input/**/L2S1-new-teacher/teacher_cot_progress.jsonl", recursive=True))
if previous and not PROGRESS_PATH.exists():
    shutil.copy(previous[0], PROGRESS_PATH)
    print(f"Continuing from {previous[0]}")

already_done = set()
if PROGRESS_PATH.exists():
    with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    already_done = {r["_idx"] for r in rows if r.get("teacher") == TEACHER_MODEL_ID}
    if len(already_done) < len(rows):
        print(f"Ignoring {len(rows) - len(already_done):,} rows from another teacher.")
    print(f"Resuming: {len(already_done):,} entries already answered by this teacher.")

@torch.inference_mode()
def generate(batch):
    """batch: list of (messages, images) -> list of answers. Greedy, with every sampling setting given explicitly,
    so the model's own generation defaults (such as a repetition penalty) can't change it."""
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m, _ in batch]
    images = [img for _, imgs in batch for img in imgs]
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=False, temperature=None,
                         top_p=None, top_k=None, repetition_penalty=1.0)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return [s.strip() for s in processor.batch_decode(new_tokens, skip_special_tokens=True)]

def generate_safe(batch):
    """Halves the batch on out-of-memory instead of crashing the run."""
    try:
        return generate(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch) == 1:
            return [""]
        half = len(batch) // 2
        return generate_safe(batch[:half]) + generate_safe(batch[half:])

todo = sorted(i for i in subset_indices if i not in already_done)
print(f"{len(todo):,} entries left to answer.")
deadline = time.time() + TIME_LIMIT_HOURS * 3600
t0, stopped, shown = time.time(), False, 0
with open(PROGRESS_PATH, "a", encoding="utf-8") as progress_fh:
    for start in tqdm(range(0, len(todo), BATCH), desc="Answers"):
        if time.time() > deadline:
            stopped = True
            break
        idxs = todo[start:start + BATCH]
        answers = generate_safe([build_gen_messages(eligible_entries[i]) for i in idxs])
        for i, answer in zip(idxs, answers):
            progress_fh.write(json.dumps({"_idx": i, "teacher": TEACHER_MODEL_ID, "teacher_rationale": answer}) + "\n")
            if shown < 3:                      # the first few, to read before trusting the run
                e = eligible_entries[i]
                print(f"\n[{e['task']}] label: {answer_text(e)[:120]!r}\n"
                      f"  answer ({'matches' if matches_label(e, answer) else 'does NOT match'} the label): "
                      f"{answer[:400]!r}")
                shown += 1
        progress_fh.flush()

print(f"\n{'Time budget reached: stopped cleanly' if stopped else 'Generation pass complete'} after "
      f"{(time.time() - t0) / 3600:.1f} h this session.")

In [ ]:
# ── Cell 6: Keep only checked-correct answers, write the distillation manifest, and compare with the first run ──
# Re-read the whole progress log (this session and any earlier one, this teacher only) and merge each answer back
# onto every manifest entry, so the file is a drop-in replacement for train_release.json in step 2. An answer is
# kept only if it gives the labelled answer (matches_label, Cell 4); a wrong or missing one leaves teacher_rationale
# empty, and step 2 trains that entry on its plain label instead.
import pandas as pd

rationale_by_idx = {}
if PROGRESS_PATH.exists():
    with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                if row.get("teacher") == TEACHER_MODEL_ID:
                    rationale_by_idx[row["_idx"]] = row["teacher_rationale"]
print(f"Answers so far: {len(rationale_by_idx):,} / {len(subset_indices):,} targeted")

distill_manifest, per_task = [], {}
for idx, entry in enumerate(eligible_entries):
    out_entry = dict(entry)
    rationale = rationale_by_idx.get(idx, "")
    correct = bool(rationale) and matches_label(entry, rationale)
    out_entry["teacher_rationale"] = rationale if correct else ""
    distill_manifest.append(out_entry)
    if idx in rationale_by_idx:            # an empty answer (out of memory) counts as answered and not kept
        counts = per_task.setdefault(entry["task"], {"generated": 0, "kept": 0})
        counts["generated"] += 1
        counts["kept"] += correct

rows = []
for task, c in sorted(per_task.items()):
    g1, k1 = FIRST_RUN[task]
    rows.append({"task": task, "generated": c["generated"], "kept": c["kept"],
                 "kept_%": round(100 * c["kept"] / c["generated"], 1),
                 "first_run_generated": g1, "first_run_kept": k1, "first_run_kept_%": round(100 * k1 / g1, 1)})
by_task = pd.DataFrame(rows)
by_task.to_csv(OUTPUT_DIR / "cot_match_by_task.csv", index=False)
print(by_task.to_string(index=False))

with open(FINAL_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(distill_manifest, f, ensure_ascii=False, indent=2)

def md_table(df):
    cols = list(df.columns)
    return "\n".join(["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
                     + ["| " + " | ".join(str(v) for v in r) + " |" for r in df.itertuples(index=False)])

n_kept = sum(1 for e in distill_manifest if e["teacher_rationale"])
first_kept = sum(k for _, k in FIRST_RUN.values())
complete = len(rationale_by_idx) >= len(subset_indices)
config = {"task": "L2 step 1 again: teacher answers from the new 7B", "smoke_test": SMOKE_TEST,
          "teacher": TEACHER_MODEL_ID, "image_size": "full (1036 px for a 1024 px image)", "decoding": "greedy",
          "max_new_tokens": GEN_MAX_NEW_TOKENS, "prompt": "the training prompt, with no added instruction",
          "entries": "the first run's 4,000 (seed 42) minus its road-counting entries",
          "targeted": len(subset_indices), "answered": len(rationale_by_idx), "kept": n_kept,
          "first_run_kept": first_kept, "complete": complete, "manifest_entries": len(distill_manifest),
          "versions": {"torch": torch.__version__, "transformers": transformers.__version__}}
(OUTPUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUTPUT_DIR / "README.md").write_text("\n".join([
    "# L2 step 1 again: teacher answers from the new 7B", "",
    f"Teacher `{TEACHER_MODEL_ID}`, at full size, greedy, asked the plain training prompt. "
    f"{len(rationale_by_idx):,} of {len(subset_indices):,} entries answered "
    f"({'smoke test' if SMOKE_TEST else 'the first run’s 4,000 minus its road-counting entries'}); "
    f"{n_kept:,} answers match their label and are kept (first run: {first_kept:,}).", "",
    "## Kept answers by task, against the first run", "",
    "The first run (19 Oct): the old 7B, the 512-token image cap, sampling at 0.2 and a 'think step by step' "
    "instruction. Its numbers are a reference, not a controlled comparison: the teacher, image size, decoding "
    "and prompt all changed.", "",
    md_table(by_task), "",
    "## Files", "",
    "- `disasterm3_distillation_cot_train.json`: every eligible training entry, with `teacher_rationale` filled "
    "only where the answer was checked correct. Step 2 reads it through `DISTILL_MANIFEST_PATH`.",
    "- `teacher_cot_progress.jsonl`: every raw answer, with the teacher's ID.",
    "- `cot_match_by_task.csv`, `config.json`.",
]), encoding="utf-8")

print(f"\n✓ Wrote {OUTPUT_DIR}: README.md, config.json, cot_match_by_task.csv, {FINAL_MANIFEST_PATH.name}, "
      f"{PROGRESS_PATH.name}")
print(f"  Entries in the manifest: {len(distill_manifest):,}; with a checked-correct teacher answer: {n_kept:,} "
      f"(first run: {first_kept:,})")
if not complete:
    print(f"\nNOTE: not finished ({len(rationale_by_idx):,}/{len(subset_indices):,}). Attach this version's output "
          f"and run again: Cell 5 continues from its {PROGRESS_PATH.name}.")